In [5]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments


In [6]:
# Small custom dataset
texts = [
    "This movie is amazing",
    "I really loved this film",
    "This was the worst movie ever",
    "I hate this movie"
]


In [7]:

labels = [1, 1, 0, 0]  # 1 = positive, 0 = negative

# Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")


In [8]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = CustomDataset(encodings, labels)

In [9]:
# Model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Training args
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Step,Training Loss
1,0.591388
2,1.013098
3,0.892760
4,0.585676
5,0.641687
6,0.642653
7,0.604611
8,0.686437
9,0.638503
10,0.645730


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10, training_loss=0.6942543745040893, metrics={'train_runtime': 38.6773, 'train_samples_per_second': 0.517, 'train_steps_per_second': 0.259, 'total_flos': 82222204800.0, 'train_loss': 0.6942543745040893, 'epoch': 5.0})

In [10]:
# Test
from transformers import pipeline
clf = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

print(clf("This movie is fantastic"))
print(clf("This movie is terrible"))

[{'label': 'LABEL_1', 'score': 0.5296331644058228}]
[{'label': 'LABEL_0', 'score': 0.5068567991256714}]
